
### Environment setup & API key configuration
- Parsing the local `.env` configuration file to safely retrieve and set the `EIA_API_KEY` environment variable while filtering out commented lines
- Setting up the base URL endpoint for the EIA API v2


In [ ]:
import os
from pathlib import Path
import pandas as pd
import requests

# loading EIA API Key from the .env file
for line in Path(".env").read_text().splitlines():
    if "=" in line and not line.strip().startswith("#"):
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip()

api_key = os.environ.get("EIA_API_KEY")
BASE = "https://api.eia.gov/v2/"

### Helper function for chunk-by-chunk data download 
- Defining the `eia_get` function to automatically download large EIA data in 5,000-row batches until the complete dataset is collected into Pandas

In [28]:
def eia_get(route, params):
    rows, offset = [], 0
    while True:
        p = {
            **params,
            "api_key": api_key,
            "offset": offset,
            "length": 5000,
        }
        r = requests.get(BASE + route + "/data/", params=p)
        r.raise_for_status()
        resp = r.json()["response"]
        rows += resp["data"]
        offset += 5000
        if offset >= int(resp["total"]):
            break
    return pd.DataFrame(rows)


### Query parameter configuration
 - Defining the request parameters to target monthly commercial sector (`COM`) retail electricity prices and sales, sorted in descending order by period

In [29]:
eia_params = {
    "frequency": "monthly",
    "data[0]": "price",
    "data[1]": "sales",
    "facets[sectorid][]": "COM",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
}


### Data retrieval & local caching 
- Executing the request against the `electricity/retail-sales` route and saving the raw DataFrame to `Data/eia_retail_sales.csv` for local caching


In [30]:
eia_raw = eia_get("electricity/retail-sales", eia_params)

eia_raw.to_csv("Data/eia_retail_sales.csv", index=False)

### EIA dataset cleaning & regional filtering
- Selecting key EIA columns (`period`, `stateid`, `price`, `sales`) and renaming `stateid` to `state_abb` for state matching
- Converting electricity price and sales values into numeric float numbers
- Removing regional totals (like `"US"`, `"ENC"`, `"SAT"`) so the dataset only contains individual U.S. states

In [ ]:
import geopandas as gpd
import pandas as pd

# clean and standardize EIA dataset columns
eia_clean = eia_raw[["period", "stateid", "price", "sales"]].copy()
eia_clean.rename(
    columns={
        "stateid": "state_abb",
        "price": "comm_elec_price_cents_kwh",
        "sales": "comm_elec_sales_mwh",
    },
    inplace=True,
)

# convert string values to numeric numbers
eia_clean["comm_elec_price_cents_kwh"] = pd.to_numeric(
    eia_clean["comm_elec_price_cents_kwh"], errors="coerce"
)
eia_clean["comm_elec_sales_mwh"] = pd.to_numeric(
    eia_clean["comm_elec_sales_mwh"], errors="coerce"
)

# filter out regional aggregate codes (US, ENC, etc.)
regional_aggregates = [
    "US", "ENC", "ESC", "MAT", "MTN", "NEW", "PACC", "PACN", "SAT", "WNC", "WSC"
]
eia_states = eia_clean[~eia_clean["state_abb"].isin(regional_aggregates)].copy()


### Isolate most recent month
- Finding the max date period in the EIA dataset
- Keeping only the most recent month's electricity rates to avoid duplicating rows during the dataset join

In [32]:

latest_period = eia_states["period"].max()
eia_latest = eia_states[eia_states["period"] == latest_period].copy()



### Load IM3 data & create spatial GeoDataFrame
- Loading the cleaned IM3 Data Center Atlas from `Data/im3_2025_atlas_cleaned.csv`
- Converting longitude and latitude coordinates into a GeoPandas `geometry` column using the WGS84 coordinate system (EPSG:4326)

In [ ]:

# load IM3 dataset and create Spatial GeoDataFrame
im3_df = pd.read_csv("Data/im3_2025_atlas_cleaned.csv")

im3_gdf = gpd.GeoDataFrame(
    im3_df,
    geometry=gpd.points_from_xy(im3_df["lon"], im3_df["lat"]),
    crs="EPSG:4326",
)

### Spatial location join (state lookup fallback)
- Checking if the `state_abb` column exists in the IM3 GeoDataFrame
- If missing, downloading U.S. state boundary shapes and running a spatial "point-in-polygon" join to assign each data center its correct state based on map location

In [ ]:



# spatial Join: Infer 'state_abb' from Point-in-Polygon location
if "state_abb" not in im3_gdf.columns:
    # stable US States GeoJSON from GitHub
    states_url = "https://raw.githubusercontent.com/python-visualization/folium-example-data/main/us_states.json"
    states = gpd.read_file(states_url)
    states = states[["id", "geometry"]].rename(columns={"id": "state_abb"})
    
    # match coordinate reference systems
    if im3_gdf.crs != states.crs:
        states = states.to_crs(im3_gdf.crs)
        
    # perform spatial point-in-polygon join
    im3_gdf = gpd.sjoin(im3_gdf, states, how="left", predicate="within")
    if "index_right" in im3_gdf.columns:
        im3_gdf.drop(columns=["index_right"], inplace=True)



### Merge IM3 facilities with EIA state grid features
- Joining the IM3 facility locations with state commercial electricity rates and sales metrics using `state_abb` as the matching key

In [ ]:

# multimodal feature fusion merge (IM3 Facilities + EIA Grid Features)
im3_multimodal_gdf = im3_gdf.merge(eia_latest, on="state_abb", how="left")



### Calculate facility power consumption & operational cost
- Setting an industry baseline power density of 100 Watts per square foot (`WATTS_PER_SQFT`)
- Estimating hourly power draw in Megawatt-hours (`est_mwh_per_hour`) based on facility square footage (`sqft`)
- Calculating estimated hourly electricity costs in USD (`est_hourly_energy_cost_usd`) using state commercial electricity rates

In [ ]:

# calculate Data Center Energy Metrics (Power Draw, Consumption & Cost)
WATTS_PER_SQFT = 100  # Standard power density assumption

# MW Power Draw & Hourly Consumption (MWh/hr)
im3_multimodal_gdf["est_mwh_per_hour"] = (
    im3_multimodal_gdf["sqft"] * WATTS_PER_SQFT
) / 1_000_000

# hourly Electricity Cost ($/hr) based on state commercial electricity price
im3_multimodal_gdf["est_hourly_energy_cost_usd"] = (
    im3_multimodal_gdf["est_mwh_per_hour"]
    * (im3_multimodal_gdf["comm_elec_price_cents_kwh"] * 10)
)



### Summary inspection & file export
- Printing summary details including total rows, columns, and a preview of calculated energy metrics
- Exporting the merged spatial dataset as a GeoJSON file (`Data/im3_multimodal_atlas.geojson`) to preserve map coordinates
- Exporting a clean CSV file (`Data/im3_multimodal_atlas.csv`) for tabular data modeling

In [ ]:

# print verification summary
print(f"Latest EIA Snapshot Period: {latest_period}")
print(f"Final Multimodal GeoDataFrame Shape: {im3_multimodal_gdf.shape}")
print("\nSample Output with Fused Multimodal Features:")
print(
    im3_multimodal_gdf[
        [
            "id",
            "state_abb",
            "sqft",
            "comm_elec_price_cents_kwh",
            "est_mwh_per_hour",
            "est_hourly_energy_cost_usd",
            "geometry",
        ]
    ].head()
)

Latest EIA Snapshot Period: 2026-07
Final Multimodal GeoDataFrame Shape: (1147, 13)

Sample Output with Fused Multimodal Features:
         id state_abb        sqft  comm_elec_price_cents_kwh  \
0   2744301        NJ    105786.0                      19.17   
1   7805491        OH    188209.0                      14.06   
2   9474864        NC   3407194.0                      10.73   
3  13924557        IA  10962475.0                      13.27   
4  14593270        NC   5431080.0                      10.73   

   est_mwh_per_hour  est_hourly_energy_cost_usd                    geometry  
0           10.5786                  2027.91762  POINT (-74.49652 40.54426)  
1           18.8209                  2646.21854  POINT (-82.81436 40.10066)  
2          340.7194                 36559.19162  POINT (-81.54651 35.89474)  
3         1096.2475                145472.04325  POINT (-93.71172 41.51595)  
4          543.1080                 58275.48840  POINT (-81.26181 35.58877)  


In [ ]:
# save as a GeoPackage or GeoJSON to preserve spatial vector geometry metadata
im3_multimodal_gdf.to_file(
    "Data/im3_multimodal_atlas.geojson", driver="GeoJSON"
)

# save as a standard CSV for quick feature tabular modeling (drops geometry column object)
im3_multimodal_gdf.drop(columns=["geometry"]).to_csv(
    "Data/im3_multimodal_atlas.csv", index=False
)
